In [ ]:
# ============================================================
# Step 0: Install packages
# ============================================================
# We need gensim for Word Embeddings and datasets for PolEmo.
# ============================================================
%pip install datasets scikit-learn gensim huggingface_hub matplotlib


In [ ]:
# ============================================================
# Step 1: Load and prepare dataset
# ============================================================
# We download Polish reviews (PolEmo2.0) from HuggingFace and
# filter to two classes: negative (1) and positive (2).
# ============================================================
# CHEATSHEET:
#   load_dataset("name", ...)                - downloads a dataset from HuggingFace
#   pd.DataFrame(data)                       - converts data to a pandas table
#   df.isin([list])                          - filters rows based on values
# ============================================================
import datasets
from datasets import load_dataset
import pandas as pd

print("Downloading Polish reviews dataset (PolEmo2.0)...")
dataset = load_dataset("clarin-pl/polemo2-official", revision="refs/convert/parquet")

# TODO: Create pandas DataFrames from dataset["train"] and dataset["test"]
df_train = None
df_test = None

# TODO: Filter rows to keep only target 1 (negative) and 2 (positive)
# df_train = ...
# df_test = ...

# TODO: Uncomment the line below to verify
# print(f"Training samples: {len(df_train)}, Test samples: {len(df_test)}")

# BONUS: Check if the classes are balanced using value_counts() on df_train["target"]


In [ ]:
# ============================================================
# Step 2: Train Word2Vec embedding model
# ============================================================
# Word2Vec learns vector representations of words from our
# training data. Each word gets a dense vector of vector_size
# dimensions. Words with similar meaning end up close together.
# ============================================================
# CHEATSHEET:
#   text.lower().split()                     - tokenizes text into words
#   Word2Vec(sentences, vector_size, ...)    - trains the model
#   model.wv["word"]                         - gets the word vector
#   model.vector_size                        - returns the vector dimensionality
#   window=N                                 - max distance between current and predicted word (context)
# ============================================================
from gensim.models import Word2Vec

# TODO: Tokenize training texts and train Word2Vec
# Hint: use Word2Vec(...) with vector_size=300
word2vec_model = None

# TODO: Uncomment the line below to verify
# print(f"Word2Vec training complete! Vector size: {word2vec_model.vector_size}")

In [ ]:
# ============================================================
# Step 3: Word Arithmetic (Semantic Relationships)
# ============================================================
# Word2Vec captures semantic relationships between words.
# We can perform arithmetic on word vectors, e.g.:
#   vector("król") - vector("mężczyzna") + vector("kobieta") ≈ vector("królowa")
# This works because the model learns that gender is a direction
# in vector space, and "king" minus "man" plus "woman" lands
# near "queen". Let's test this on our Polish model!
# ============================================================
# CHEATSHEET:
#   model.wv.most_similar(positive, negative) - finds closest words to the result
#   "word" in model.wv                        - checks if word is in vocabulary
#   model.wv.similarity(w1, w2)               - cosine similarity between words
# ============================================================

# TODO: Test word arithmetic - "król" - "mężczyzna" + "kobieta" = ?
# Hint: use word2vec_model.wv.most_similar(positive=[...], negative=[...], topn=5)
result = None

# TODO: Uncomment to see the result
# print('król - mężczyzna + kobieta =')
# for word, score in result:
#     print(f"  {word:20s} (similarity: {score:.4f})")

# NOTE: Results depend heavily on training data. Our model learned
# from Polish product reviews, so some classic analogies may not work
# perfectly. The vocabulary is limited to words from the reviews.

# YOUR IDEAS: Try your own word arithmetic!


In [ ]:
# ============================================================
# Step 4: Sentence to Vector transformation (Averaging)
# ============================================================
# We convert each text into one vector by averaging token vectors.
# ============================================================
# CHEATSHEET:
#   text.lower().split()                     - splits text into a list of words
#   word in model.wv                         - checks if word exists in vocabulary
#   model.wv[word]                           - gets the word vector
#   model.vector_size                        - returns the vector dimensionality
#   np.mean(list, axis=0)                    - calculates the average of vectors
#   np.zeros(N)                              - creates a zero vector of size N
#   np.array(list)                           - converts a list of arrays into a matrix
# ============================================================
import numpy as np

def get_sentence_vector(text, model):
    # TODO: Split the text into lowercased tokens
    tokens = None
    
    # TODO: Create a list of vectors for each word in tokens if it exists in model.wv
    vectors = None
    
    # TODO: If the vectors list is empty, return a zero vector matching model dimensions
    # Hint: use model.vector_size
    
    # TODO: Return the mathematical mean of the vectors array along axis 0
    return None

# Assign the model for use in subsequent steps
embedding_model = word2vec_model

print("Vectorizing training data...")
# TODO: Apply the function to all texts in df_train to create X_train_emb
# Hint: use np.array([...]) and get_sentence_vector(text, embedding_model)
X_train_emb = None

print("Vectorizing test data...")
# TODO: Apply the function to all texts in df_test to create X_test_emb
X_test_emb = None

# IMPORTANT: Do not forget to assign targets!
y_train = df_train["target"]
y_test = df_test["target"]

# TODO: Uncomment to verify
# print("Embeddings matrix shape (train):", X_train_emb.shape)

# BONUS: Test what happens if you pass an empty string or unknown words


In [ ]:
# ============================================================
# Step 5: Train Logistic Regression and Evaluate
# ============================================================
# We use the dense embeddings matrix to train the model.
# ============================================================
# CHEATSHEET:
#   LogisticRegression(max_iter=N)           - initializes the model
#   classifier.fit(X, y)                     - trains the model
#   classifier.predict(X)                    - predicts labels for new data
#   accuracy_score(y_true, y_pred)           - calculates accuracy percentage
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# TODO: Initialize LogisticRegression with max_iter=1000
classifier = None

# TODO: Train the classifier using X_train_emb and y_train


# TODO: Predict the results for X_test_emb
predictions = None

# TODO: Calculate the accuracy
accuracy = None

# TODO: Uncomment to verify
# print(f"Model accuracy (Word Embeddings): {accuracy * 100:.2f}%")
# print()
# print(classification_report(y_test, predictions, target_names=["negative", "positive"]))

# BONUS: Test the model on your own custom review (remember to vectorize it first!)


In [ ]:
# ============================================================
# Step 6: Compare with Random Forest
# ============================================================
# Because our embeddings matrix is dense (no zeros) and compact
# (vector_size columns), we can test tree-based models easily.
# ============================================================
# CHEATSHEET:
#   RandomForestClassifier(n_estimators=N)   - initializes Random Forest
#   model.fit(X, y)                          - trains the model
#   model.predict(X)                         - predicts labels
#   accuracy_score(y_true, y_pred)           - calculates accuracy percentage
# ============================================================
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest...")

# TODO: Initialize RandomForestClassifier with n_estimators=100 and random_state=42
classifier_rf = None

# TODO: Train the Random Forest classifier


# TODO: Predict the results and calculate accuracy for Random Forest
predictions_rf = None
accuracy_rf = None

# TODO: Uncomment to verify
# print(f"Logistic Regression Accuracy: {accuracy * 100:.2f}%")
# print(f"Random Forest Accuracy:       {accuracy_rf * 100:.2f}%")

# BONUS: Try changing n_estimators to 10 or 500 and see how accuracy and training time change.


In [ ]:
# ============================================================
# Step 7: The Flaw of Averaging (Loss of Sequence)
# ============================================================
# Averaging vectors destroys word order. "A hit B" and "B hit A"
# can result in almost the same vector and prediction.
# ============================================================
# CHEATSHEET:
#   get_sentence_vector(text, model)         - converts text to a vector
#   np.allclose(arr1, arr2)                  - checks if arrays are nearly equal
# ============================================================
import numpy as np

# YOUR IDEAS: Think of your own sentence pairs where swapping words changes the meaning.
sentence_1 = "tani telefon i drogi abonament"
sentence_2 = "drogi telefon i tani abonament"

# TODO: Vectorize both sentences using get_sentence_vector
vec_1 = None
vec_2 = None

# TODO: Check if vec_1 and vec_2 are mathematically identical
are_equal = None

# NOTE: We use np.allclose() instead of np.array_equal() because floating-point
# addition is not commutative at machine precision. np.mean() sums vectors in
# token order, so two sentences with the same words but different order produce
# results that differ by tiny rounding errors (~1e-7). np.allclose() ignores
# these artefacts and confirms the vectors are semantically identical - which is
# exactly the flaw we want to demonstrate.

# TODO: Uncomment to verify the flaw
# print(f"Sentence 1: '{sentence_1}'")
# print(f"Sentence 2: '{sentence_2}'")
# print(f"Are vectors nearly identical? {are_equal}")

# BONUS: Try writing a sentence with a negation and compare predictions.

In [ ]:
# ============================================================
# Step 8: The Out of Vocabulary (OOV) Problem
# ============================================================
# In Word2Vec, unknown tokens are skipped entirely.
# If all words are unknown, our function returns a zero vector.
# ============================================================
# CHEATSHEET:
#   get_sentence_vector(text, model)         - converts text to a vector
#   np.count_nonzero(arr)                    - counts non-zero elements in array
#   classifier.predict_proba(X)              - returns confidence for each class
# ============================================================
gibberish_text = "blarghhhh qwertyuiop xdxdxdxd"

# TODO: Vectorize the gibberish_text
vec_oov = None

# TODO: Count how many non-zero values are in the resulting vector
non_zero_count = None

# TODO: Uncomment to verify
# print(f"Text: '{gibberish_text}'")
# print(f"First 5 dimensions of the vector: {vec_oov[:5]}")
# print(f"Number of non-zero values: {non_zero_count} out of {embedding_model.vector_size}")

# NOTE: With Word2Vec this is a zero vector for pure gibberish.
# The classifier still returns a class, but confidence can be misleading.

# TODO: Uncomment to see the model's confidence on gibberish
# proba = classifier.predict_proba([vec_oov])[0]
# print(f"\nModel prediction confidence: NEG={proba[0]*100:.1f}% POS={proba[1]*100:.1f}%")

In [ ]:
# ============================================================
# Step 9: The Black Box Problem (Interpretability loss)
# ============================================================
# In TF-IDF, each column was a specific word. Here, columns are
# abstract geometric dimensions. We cannot map weights to words anymore.
# ============================================================
# CHEATSHEET:
#   classifier.coef_[0]                      - gets model weights array
#   len(arr)                                 - returns the length of the array
# ============================================================

# TODO: Extract the learned weights from our trained LogisticRegression model
weights = None

# TODO: Uncomment to see the problem
# print(f"Total number of features (weights): {len(weights)}")
# print("Top 5 weights learned by the model:")
# 
# for i in range(5):
#     print(f"Dimension {i}: Weight = {weights[i]:.4f}")

# NOTE: We know Dimension 0 has a specific weight, but what does 
# Dimension 0 mean? Is it 'happiness'? Is it 'cars'? We don't know.
# This is the cost of moving from TF-IDF to Deep Learning embeddings.

# BONUS: Compare this mental model with yesterday's TF-IDF script 
# where you could clearly print the most negative and positive words.

In [ ]:
# ============================================================
# Step 10: Visualizing Word Vectors in 2D
# ============================================================
# Word vectors live in 300 dimensions - impossible to plot directly.
# PCA and t-SNE reduce them to 2D so we can see semantic clusters.
# PCA is linear and fast; t-SNE is non-linear and better at
# preserving local neighborhood structure.
# ============================================================
# CHEATSHEET:
#   PCA(n_components=2)                      - initializes PCA reducer
#   TSNE(n_components=2, random_state=42)    - initializes t-SNE reducer
#   reducer.fit_transform(matrix)            - reduces dimensionality
#   model.wv[word]                           - gets the word vector
#   plt.scatter(x, y)                        - scatter plot
#   plt.annotate(label, xy=(x, y))           - adds text label to a point
# ============================================================
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import numpy as np

words = [
    "dobry", "zły", "świetny", "słaby", "piękny",
    "szybki", "wolny", "drogi", "tani", "nowy",
    "produkt", "telefon", "laptop", "kamera", "sklep",
    "polecam", "odradzam", "kupić", "zwrócić", "reklamacja",
]

# TODO: Filter words to keep only those present in word2vec_model's vocabulary
# Hint: use "word" in word2vec_model.wv
words_in_vocab = None

# TODO: Build a matrix of word vectors for words_in_vocab
word_matrix = None

# TODO: Reduce dimensions to 2 using PCA
coords_pca = None

# TODO: Reduce dimensions to 2 using t-SNE
# Hint: use perplexity=5 (must be smaller than the number of words)
coords_tsne = None

# TODO: Uncomment to plot both visualizations side by side
# fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# for ax, coords, title in [
#     (axes[0], coords_pca, "PCA"),
#     (axes[1], coords_tsne, "t-SNE"),
# ]:
#     ax.scatter(coords[:, 0], coords[:, 1], alpha=0.7)
#     for i, word in enumerate(words_in_vocab):
#         ax.annotate(word, (coords[i, 0], coords[i, 1]), fontsize=9)
#     ax.set_title(f"Word Vectors ({title})")
#     ax.set_xlabel("Dimension 1")
#     ax.set_ylabel("Dimension 2")
# plt.tight_layout()
# plt.show()

# YOUR IDEAS: Try different word groups to reveal semantic clusters!

In [ ]:
# ============================================================
# Step 11 (BONUS): FastText — comparison with Word2Vec
# ============================================================
# FastText is a word embedding model from Facebook Research.
# Unlike Word2Vec, FastText handles unknown words (OOV) thanks
# to breaking words into character n-grams (subword information).
#
# Trade-off: FastText requires downloading a large pre-trained
# model (~6 GB for Polish), while Word2Vec trains on your data.
# ============================================================
# CHEATSHEET:
#   urllib.request.urlretrieve(url, path)    - downloads a file from url
#   load_facebook_model(path)                - loads FastText .bin.gz file
#   model.wv.similarity(w1, w2)             - cosine similarity between words
# ============================================================
# WARNING: The model file (cc.pl.300.bin.gz) weighs ~6 GB and
# requires significant RAM. Run only if you have resources.
#
# NOTE Using pre-trained models is called Transfer Learning.
# ============================================================

# import os
# import urllib.request
# from gensim.models.fasttext import load_facebook_model

# model_path = "cc.pl.300.vec.gz" # lihghter
# # model_path = "cc.pl.300.bin.gz"

# # TODO: Download the FastText model if it doesn't exist on disk
# # Hint: use os.path.exists() and urllib.request.urlretrieve()


# # TODO: Load the FastText model using load_facebook_model()
# fasttext_model = None

# # TODO: Check similarity between "auto" and "samochód"
# # Hint: use fasttext_model.wv.similarity(w1, w2)

# # TODO: Re-vectorize data with FastText and compare accuracy with Word2Vec
# # Hint: use get_sentence_vector with fasttext_model, then train a new LogisticRegression
# X_train_ft = None
# X_test_ft = None

# # TODO: Compare OOV behavior between Word2Vec and FastText
# # Word2Vec returns a zero vector for unknown words; FastText does not.
# gibberish = "blarghhhh qwertyuiop xdxdxdxd"
# # vec_w2v = get_sentence_vector(gibberish, word2vec_model)
# # vec_ft = get_sentence_vector(gibberish, fasttext_model)
# # print(f"Word2Vec non-zero: {np.count_nonzero(vec_w2v)} / {word2vec_model.vector_size}")
# # print(f"FastText non-zero: {np.count_nonzero(vec_ft)} / {fasttext_model.vector_size}")